In [1]:
import polars as pl
from tqdm.auto import tqdm
from typing import Dict, List, Optional
import pandas as pd
import networkx as nx
import helen0822 as hsds

In [2]:
# Desired columns, in desired order
COL_ORDERS = ['hash', 'source', 'from', 'to', 'value', 'symbol', 'sub_type']#, 'transaction_fee', 'type', 'ETH_adj']
# Theses last 3 columns comes from the etc_traced file only

In [3]:
def load_data(path: str, src_hash: str) -> Dict[str, pl.DataFrame]:
    """Load Excel files and return a Dict with all of them as DataFrames"""
    data = {}
    
    filenames = {
        "main": f"{path}/TX_Main_Pool_or_Add_{src_hash}_formatted.xlsx",
        "internal": f"{path}/TX_Internes_Pool_or_Add_{src_hash}_formatted.xlsx",
        "erc_full": f"{path}/TX_ERCFull_Pool_or_Add_{src_hash}_formatted.xlsx",
        "erc_traced": f"{path}/TX_ERC_Traced_Cleaned_{src_hash}.xlsx",
        "erc": f"{path}/TX_ERC_Pool_or_Add_{src_hash}_formatted.xlsx" #new
    }

    for name, filename in filenames.items():
        try:
            data[name] = pl.read_excel(filename)
            print(f"{name.capitalize()} DataFrame loaded successfully. Shape: {data[name].shape}")
        except FileNotFoundError:
            print(f"File for {name} not found: {filename}")
            data[name] = pl.DataFrame()  # Empty DataFrame if file not found

    return data

In [4]:
# Debugging
# import pandas as pd
# pd.DataFrame(data['erc_traced']['final_value']).isnull().all()

In [5]:
# # test
# # The 9 folders I am digging in:

# hashes_to_parse = [
#     ["./Data/ABCD 24-03-17 01", "0xe96df8f5ef1a8790415068c798765b07d57643bd"], #I re-named this folder.
#     ["./Data/AFY", "0xf43e889444e95a0429c32b0b601dc16edf90fdbf"],
#     ["./Data/BRETT 24-03-20 3j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"],
#     ["./Data/BRETT 24-03-28 2j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"],
#     ["./Data/ECT", "0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758"],
#     ["./Data/ORCH", "0x61738277ff9b92e91b8ee02c0b73e8369b79ccfc"],
#     ["./Data/PEPE UniV2 24-04-05 all", "0x76fc8dbf2022ee75612cb74ec80caf6b3ccffb23"],
#     ["./Data/TOSHI 24-03-24 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"],
#     ["./Data/TOSHI 24-04-28 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"],
# ]

# for path, src_hash in hashes_to_parse:
#     data = load_data(path, src_hash)
#     if not (pd.DataFrame(data['erc_traced']['final_value']).isnull().all()[0]):
#         print("not all null here")

In [6]:
def process_main_df(df: pl.DataFrame) -> pl.DataFrame:    
    # Process the DataFrame
    df = (
        df
        .select(['hash', 'from', 'to', 'value'])  # Select initial columns
        .with_columns([
            # pl.lit(None).cast(pl.Float64).alias("transaction_fee"),  # cast 'fee'
            pl.col("value").cast(pl.Float64),  # Cast 'value'
            pl.lit("ETH").cast(pl.Utf8).alias("symbol"),  # Add 'symbol' column
            pl.lit("main").cast(pl.Utf8).alias("source"),  # Add 'source' column
            pl.lit("main").cast(pl.Utf8).alias("sub_type"),  # Add 'sub_type' column
            # pl.lit(None).cast(pl.Utf8).alias("type"),  # Add 'type' column with None value
            # pl.lit(None).cast(pl.Float64).alias("ETH_adj"),  # Add 'ETH_adj' column with None value
        ])
    )
    
    # Ensure columns are in the specified order
    df = df.select(COL_ORDERS)
    
    print("Processed Main DataFrame shape:", df.shape)
    return df

In [7]:
# test

# df = process_main_df(data['main'])
# df.schema

In [8]:
def process_internal_df(df: pl.DataFrame) -> pl.DataFrame:
    # Process the DataFrame
    df = (
        df
        .select(['hash', 'from', 'to', 'value', 'callType'])  # relevant columns
        .with_columns([
            pl.col("callType").cast(pl.Utf8).alias("sub_type"),  
            pl.col("value").cast(pl.Float64).alias("value"),  
            pl.lit("ETH").cast(pl.Utf8).alias("symbol"),  
            pl.lit("internal").cast(pl.Utf8).alias("source"),  # Add 'source' column with constant value 'internal'
            # pl.lit(None).cast(pl.Float64).alias("transaction_fee"),  # Add 'transaction_fee' column with None value
            # pl.lit(None).cast(pl.Utf8).alias("type"),  # Add 'type' column with None value
            # pl.lit(None).cast(pl.Float64).alias("ETH_adj")  # Add 'ETH_adj' column with None value
        ])
    )
    
    # Ensure columns are in the specified order
    df = df.select(COL_ORDERS)
    
    # Print the shape of the processed DataFrame
    print("Processed Internal DataFrame shape:", df.shape)
    
    # Return the processed DataFrame
    return df

In [9]:
# test

# df = process_internal_df((data['internal']))
# df.schema

In [10]:
def process_erc_full_df(df: pl.DataFrame) -> pl.DataFrame:
    """Process the ERC Full DataFrame."""
    df = (
        df
        .select(['hash', 'from', 'to', 'tokenSymbol', 'value', 'tx_type'])  # relevant columns
        .with_columns([
            pl.col("tokenSymbol").cast(pl.Utf8).alias("symbol"),  
            pl.col("tx_type").cast(pl.Utf8).alias("sub_type"), 
            pl.col("value").cast(pl.Float64).alias("value"),  
            pl.lit("erc_full").cast(pl.Utf8).alias("source"), 
            # pl.lit(None).cast(pl.Float64).alias("transaction_fee"), 
            # pl.lit(None).cast(pl.Utf8).alias("type"),  
            # pl.lit(None).cast(pl.Float64).alias("ETH_adj")  
        ])
    )
    
    # Ensure columns are in the specified order
    df = df.select(COL_ORDERS)
    
    # Print the shape of the processed DataFrame
    print("Processed ERC Full DataFrame shape:", df.shape)
    
    # Return the processed DataFrame
    return df


In [11]:
# test

# df = process_erc_full_df((data['erc_full']))
# df.schema

In [12]:
# data['erc_traced'].schema


In [13]:
def process_erc_traced_df(df: pl.DataFrame) -> pl.DataFrame:
    """Process the ERC Traced DataFrame."""
    df = (
        df
        .select(['Tx hash', 'Tx type', 'final_value', 'Fee value'])
        .with_columns([
            pl.col("Tx hash").cast(pl.Utf8).alias("hash"),  
            pl.col("Tx type").cast(pl.Utf8).alias("type"),  
            pl.col("final_value").cast(pl.Float64).alias("ETH_adj"),  
            pl.col("Fee value").cast(pl.Float64).alias("transaction_fee"), 
             ])
    )
    
    # Ensure columns are in the specified order
    df = df.select(["hash", "type", "ETH_adj", "transaction_fee"])
    
    # Print the shape of the processed DataFrame
    print("Processed ERC Traced DataFrame shape:", df.shape)
    
    # Return the processed DataFrame
    return df


In [14]:
# test

# df = process_erc_traced_df((data['erc_traced']))
# df.schema

In [15]:
def process_erc_df(df: pl.DataFrame) -> pl.DataFrame:
    """Process the ERC DataFrame."""
    df = (
        df
        .select(['hash', 'from', 'to', 'tokenSymbol', 'value', 'tx_type', 'transactionFee'])  # Select relevant columns
        .with_columns([
            pl.col("tokenSymbol").cast(pl.Utf8).alias("symbol"),  
            pl.col("tx_type").cast(pl.Utf8).alias("sub_type"),  
            pl.col("value").cast(pl.Float64).alias("value"),  
            # pl.col("transactionFee").cast(pl.Float64).alias("transaction_fee"),  # I am only taking 'transaction_fee' from erc_traced file
            pl.lit("erc").cast(pl.Utf8).alias("source"), 
            # pl.lit(None).cast(pl.Utf8).alias("type"),  
            # pl.lit(None).cast(pl.Float64).alias("ETH_adj")  
        ])
    )
    
    # Ensure columns are in the specified order
    df = df.select(COL_ORDERS)
    
    # Print the shape of the processed DataFrame
    print("Processed ERC DataFrame shape:", df.shape)
    
    # Return the processed DataFrame
    return df


In [16]:
# test

# df = process_erc_df((data['erc']))
# df.schema

In [17]:
# dfs = {
#     'main': process_main_df(data['main']),
#     'internal': process_internal_df(data['internal']),
#     'erc_full': process_erc_full_df(data['erc_full']),
#     'erc_traced': process_erc_traced_df(data['erc_traced']),
#     'erc': process_erc_df(data['erc'])
# }
# # final_df = concatenate_and_process_dataframes(processed_dfs)
# dfs_aligned = dfs

In [18]:
# [print(df.schema) for df in dfs_aligned.values()]
    
# df = (
#     pl.concat([dfs_aligned['main'], dfs_aligned['internal'], dfs_aligned['erc_full'], dfs_aligned['erc']])
#     .sort(["hash", "source", "from", "to"])
# )
# print("Concatenated DataFrame shape before join:", df.shape)

# if not dfs_aligned['erc_traced'].is_empty():
#     df = df.join(dfs_aligned['erc_traced'], on="hash", how="left")
#     print("DataFrame shape after join with ERC Traced:", df.shape)


In [19]:
# df.schema

In [20]:
def concatenate_and_process_dataframes(dfs: Dict[str, pl.DataFrame]) -> pl.DataFrame:
    """Concatenate and process DataFrames."""
    # Align columns before concatenation
    dfs_aligned = dfs
    
    df = (
        pl.concat([dfs_aligned['main'], dfs_aligned['internal'], dfs_aligned['erc_full'], dfs_aligned['erc']])
        .sort(["hash", "source", "from", "to"])
    )
    print("Concatenated DataFrame shape before join:", df.shape)
    
    if not dfs_aligned['erc_traced'].is_empty():
        df = df.join(dfs_aligned['erc_traced'], on="hash", how="left")
        print("DataFrame shape after join with ERC Traced:", df.shape)
    
    print("DataFrame shape:", df.shape)

    # Remove duplicate rows, maybe it is unnecessary
    df_unique = df.unique()
    
    print("DataFrame shape after removing duplicates:", df_unique.shape)
    return df_unique

In [21]:
def main() -> pl.DataFrame:
    """Process all data and return a unique final DataFrame."""
    hashes_to_parse = [
        ["./Data/ABCD 24-03-17 01", "0xe96df8f5ef1a8790415068c798765b07d57643bd"],
        ["./Data/AFY", "0xf43e889444e95a0429c32b0b601dc16edf90fdbf"],
        ["./Data/BRETT 24-03-20 3j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"],
        ["./Data/BRETT 24-03-28 2j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"],
        ["./Data/ECT", "0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758"],
        ["./Data/ORCH", "0x61738277ff9b92e91b8ee02c0b73e8369b79ccfc"],
        ["./Data/PEPE UniV2 24-04-05 all", "0x76fc8dbf2022ee75612cb74ec80caf6b3ccffb23"],
        ["./Data/TOSHI 24-03-24 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"],
        ["./Data/TOSHI 24-04-28 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"],
    ]
    
    all_dfs = []
    for path, src_hash in tqdm(hashes_to_parse):
        data = load_data(path, src_hash)
        processed_dfs = {
            'main': process_main_df(data['main']),
            'internal': process_internal_df(data['internal']),
            'erc_full': process_erc_full_df(data['erc_full']),
            'erc_traced': process_erc_traced_df(data['erc_traced']),
            'erc': process_erc_df(data['erc'])
        }
        final_df = concatenate_and_process_dataframes(processed_dfs)
        all_dfs.append(final_df)
    
    if all_dfs:
        return pl.concat(all_dfs)
    else:
        return pl.DataFrame(columns=COL_ORDERS)

In [51]:
def write_parquet(df: pl.DataFrame, filename: str) -> None:
    """Write the final DataFrame to a Parquet file."""
    df.write_parquet(filename)
    print(f"Data written to {filename}")

In [23]:
final_df = main()
write_parquet(final_df, "./data/final_df.pqt")


 11%|█         | 1/9 [00:00<00:03,  2.40it/s]

Main DataFrame loaded successfully. Shape: (19, 50)
Internal DataFrame loaded successfully. Shape: (1297, 17)
Erc_full DataFrame loaded successfully. Shape: (697, 17)
Erc_traced DataFrame loaded successfully. Shape: (3, 15)
Erc DataFrame loaded successfully. Shape: (35, 17)
Processed Main DataFrame shape: (19, 7)
Processed Internal DataFrame shape: (1297, 7)
Processed ERC Full DataFrame shape: (697, 7)
Processed ERC Traced DataFrame shape: (3, 4)
Processed ERC DataFrame shape: (35, 7)
Concatenated DataFrame shape before join: (2048, 7)
DataFrame shape after join with ERC Traced: (2048, 10)
DataFrame shape: (2048, 10)
DataFrame shape after removing duplicates: (980, 10)
Main DataFrame loaded successfully. Shape: (8614, 45)
Internal DataFrame loaded successfully. Shape: (145790, 17)
Erc_full DataFrame loaded successfully. Shape: (39758, 17)
Erc_traced DataFrame loaded successfully. Shape: (8584, 17)
Erc DataFrame loaded successfully. Shape: (28248, 18)
Processed Main DataFrame shape: (86

 22%|██▏       | 2/9 [00:10<00:43,  6.21s/it]

DataFrame shape after removing duplicates: (191032, 10)
Main DataFrame loaded successfully. Shape: (10542, 50)
Internal DataFrame loaded successfully. Shape: (171187, 17)
Erc_full DataFrame loaded successfully. Shape: (43368, 17)
Erc_traced DataFrame loaded successfully. Shape: (10199, 19)
Erc DataFrame loaded successfully. Shape: (21176, 17)
Processed Main DataFrame shape: (10542, 7)
Processed Internal DataFrame shape: (171187, 7)
Processed ERC Full DataFrame shape: (43368, 7)
Processed ERC Traced DataFrame shape: (10199, 4)
Processed ERC DataFrame shape: (21176, 7)
Concatenated DataFrame shape before join: (246273, 7)
DataFrame shape after join with ERC Traced: (246273, 10)
DataFrame shape: (246273, 10)


 33%|███▎      | 3/9 [00:29<01:12, 12.09s/it]

...

 89%|████████▉ | 8/9 [01:28<00:10, 10.39s/it]

DataFrame shape after removing duplicates: (250378, 10)
Main DataFrame loaded successfully. Shape: (2193, 50)
Internal DataFrame loaded successfully. Shape: (69387, 17)
Erc_full DataFrame loaded successfully. Shape: (16840, 17)
Erc_traced DataFrame loaded successfully. Shape: (1920, 17)
Erc DataFrame loaded successfully. Shape: (4859, 17)
Processed Main DataFrame shape: (2193, 7)
Processed Internal DataFrame shape: (69387, 7)
Processed ERC Full DataFrame shape: (16840, 7)
Processed ERC Traced DataFrame shape: (1920, 4)
Processed ERC DataFrame shape: (4859, 7)


100%|██████████| 9/9 [01:31<00:00, 10.19s/it]

Concatenated DataFrame shape before join: (93279, 7)
DataFrame shape after join with ERC Traced: (93279, 10)
DataFrame shape: (93279, 10)
DataFrame shape after removing duplicates: (66488, 10)


Data written to ./data/final_df.pqt


In [24]:
final_df.schema


Schema([('hash', String),
        ('source', String),
        ('from', String),
        ('to', String),
        ('value', Float64),
        ('symbol', String),
        ('sub_type', String),
        ('type', String),
        ('ETH_adj', Float64),
        ('transaction_fee', Float64)])

In [25]:
final_df.head()

hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee
str,str,str,str,f64,str,str,str,f64,f64
"""0x098a3516531805b45517dfe8a73b…","""erc_full""","""0x4ed4E862860beD51a9570b96d89a…","""0xe3900437316cf425f7ea297972cC…",1.752393e6,"""Airdrop: degen.gifts/?claim""","""ERC20""",null,null,null
"""0x23fb47e6dd44855611f66de328d8…","""erc_full""","""0x7d9CE55D54FF3FEddb611fC63fF6…","""0xBfACaE089DD75D5a6073d96165A9…",494611.0,"""Claim at: claim.fungi-airdrop.…","""ERC20""",null,null,null
"""0x78411429aed55438fa64f20ef015…","""internal""","""0xfcd3842f85ed87ba2889b4d35893…","""0xe96df8f5ef1a8790415068c79876…",0.0,"""ETH""","""call""",null,null,null
"""0x60306c03c3f84eaf2fd328c9d68c…","""erc_full""","""0xAC1Bd2486aAf3B5C0fc3Fd868558…","""0xE5400a3500554BbC8216b3a0A3aE…",1.7552393e7,"""Claim: toshi-thecat.com/?claim""","""ERC20""",null,null,null
"""0xd11e852a886a52519e0d47a7c072…","""erc_full""","""0x09350F89e2D7B6e96bA730783c2d…","""0xC75A963842c07B44ba87dbE3B576…",8.880888e6,"""BENTO""","""ERC20""","""transfer token autre que le po…",null,4.35163512e8


In [26]:
# checking that the transaction fee is not all null
# final_df['transaction_fee'].is_null().all()

In [2]:
# ignore the hash that the 'to' is missing
final_df = final_df.filter(~pl.col('to').is_null())

# make it a pandas dataframe
df_pandas = final_df.to_pandas()


NameError: name 'final_df' is not defined

In [28]:
# Filtrar pelas transacoes onde a gente tem o ETH_adj:

df_pandas["hash"] = df_pandas["hash"].str.lower()
df_pandas["to"] = df_pandas["to"].str.lower()
df_pandas["from"] = df_pandas["from"].str.lower()

df_cleaned = df_pandas[df_pandas["ETH_adj"].notna()]

columns_to_check = ['hash', 'from', 'to', 'value', 'symbol', 'sub_type', 'type', 'ETH_adj', 'transaction_fee']

df_cleaned = df_cleaned.drop_duplicates(subset=columns_to_check, keep='first')

hashes = df_cleaned["hash"].unique()

df_grouped = df_cleaned.groupby("hash")

In [29]:
df_cleaned.columns

Index(['hash', 'source', 'from', 'to', 'value', 'symbol', 'sub_type', 'type',
       'ETH_adj', 'transaction_fee'],
      dtype='object')

In [30]:
n_hashes = len(hashes)
n_hashes

73341

In [31]:
# In theory, I am recovering 73,341 transactions.

In [32]:
# chosen_hash = hashes[5]
# chosen_group = df_grouped.get_group(chosen_hash)
# #type(chosen_group)
# chosen_group

In [33]:
# It looks like ERC is not contained in ERC_full!

In [34]:
# # Example
# chosen_hash = hashes[5]

# chosen_group = df_grouped.get_group(chosen_hash)

# source_node = hsds.find_source_node(chosen_group, chosen_hash)

In [35]:
# # Adapted from Jackys utils:
# # Hash Renaming
# import numpy as np
# import string
# import itertools
    
# hashes = chosen_group[["from", "to"]].values.flatten()
# _, idx = np.unique(hashes, return_index=True)

# # Generate more unique names if needed
# letters = list(string.ascii_letters)
# if len(hashes[idx]) > len(letters):
#     letters.extend([''.join(i) for i in itertools.product(string.ascii_letters, repeat=2)])

# # map hashes to letters
# renaming_dict = dict(
#     zip(
#         hashes[idx], 
#         letters[:len(hashes[idx])]
#     )
# )
    
# #SOURCE = renaming_dict[source_node]
    
# chosen_group["from"] = chosen_group["from"].map(renaming_dict)
# chosen_group["to"] = chosen_group["to"].map(renaming_dict)

# G = create_graph_from_dataframe(chosen_group)

In [36]:
# # # Adapted from Jackys utils:
# # This is in a function below

# import numpy as np
# import string
# import itertools

# # Flatten and get unique hash values
# hashes = chosen_group[["from", "to"]].values.flatten()
# _, idx = np.unique(hashes, return_index=True)

# # Generate more unique names if needed
# letters = list(string.ascii_letters)
# if len(hashes[idx]) > len(letters):
#     letters.extend([''.join(i) for i in itertools.product(string.ascii_letters, repeat=2)])

# # Map hashes to letters
# renaming_dict = dict(
#     zip(
#         hashes[idx], 
#         letters[:len(hashes[idx])]
#     )
# )

# SOURCE = renaming_dict[source_node]

# # Rename using .loc to avoid SettingWithCopyWarning
# chosen_group.loc[:, "from"] = chosen_group["from"].map(renaming_dict)
# chosen_group.loc[:, "to"] = chosen_group["to"].map(renaming_dict)

# # Create graph from the renamed DataFrame
# G = create_graph_from_dataframe(chosen_group)


In [37]:
# Exemplo de um grafo com mais de uma componente conexa
# # chosen_hash = "0x2937cb043ce362488319fbdccc6ab276c9a91b9cdaca4b1006414838109edf63"
# # https://etherscan.io/tx/0x2937cb043ce362488319fbdccc6ab276c9a91b9cdaca4b1006414838109edf63
# # 
# # From
# # Null: 0x000...000
# # To
# # Null: 0x00...dEaD
# # For
# # 1,081.980828765845224703

# # Uniswap V2 (UNI-V2)

In [38]:
len(hashes)

73341

In [39]:
for chosen_hash in hashes:
    chosen_group = df_grouped.get_group(chosen_hash)
    if len(chosen_group) < 10:
        print(f"This hash's dataframe has less than 10 rows:", chosen_hash)

This hash's dataframe has less than 10 rows: 0x0b34d5f5ff324fb90d3b7f065c7f314588191b97606ce9835272d3b248ccfcd4
This hash's dataframe has less than 10 rows: 0x28158bf7fea5d2c9bb6048d0817e246977c4b01f0d333e897226bfcca5a8deca
This hash's dataframe has less than 10 rows: 0x2c29b85a5b460c74efab2d33b8b9a521d45e6b522a5f15e4d26bf47f3c96b0c7
This hash's dataframe has less than 10 rows: 0x12a1e870091fbe0f2e89bb974633a61ca9f3a6b3e55398bde6d1658c89585370
This hash's dataframe has less than 10 rows: 0x475767c483d39f020956b141274584420237dbf69f68c7fb8bf9d3f1016b46f1
This hash's dataframe has less than 10 rows: 0x732e7646e65be72880c696cbee05dc74b04108bdc44d9b1b068b9025029524eb
This hash's dataframe has less than 10 rows: 0xc63f4bb2c97246660a7c523fab8b586eb4fa805324ea76f9e90499983125c2e0
This hash's dataframe has less than 10 rows: 0x90a53cb17efd47917ba72a9c89e97d260374b1ab63770e41ca7d5f39078dba88
This hash's dataframe has less than 10 rows: 0xb83ed86f80eee4a500d4afe3027ebac3e0be2443fffb392a8645a4351

In [40]:
import importlib
import helen0822 as hsds
import utils as ut #Jacky's utils

importlib.reload(hsds)

chosen_hash = "0x63822ce79d56b6150e4486c5069bed3624de3a5beb9f8fc741cdbf7a142359d7"

chosen_group = df_grouped.get_group(chosen_hash)

source_node = hsds.find_source_node(chosen_group)

chosen_group, source_node = hsds.hashes_to_letters(chosen_group, source_node)

In [41]:
# Really good example with duplicate transactions:
# chosen_hash = "0x63822ce79d56b6150e4486c5069bed3624de3a5beb9f8fc741cdbf7a142359d7"

In [42]:
graph, fig = hsds.draw_base_graph_four(chosen_group, source_node, create_using=nx.MultiDiGraph)
fig

# Tasks: # OUTDATED
# Trocar WETH / ETH por WETH - decided to incorporate them both
# Print symbol and value on the edges like Jacky's - DONE
# Check if this graph indeed represents the chosen_group dataframe 
# Run the algos and debug it
# Run the algorithms on this graph and compare the result with what I wanted to get - DONE
# Create new colum in the dataframe - DONE
# Finally, compare with AMF results

In [43]:
ETH_value_by_AMF = chosen_group["ETH_adj"]
ETH_value_by_AMF

67588     0.485128
108396    0.485128
118220    0.485128
122932    0.485128
125037    0.485128
132451    0.485128
Name: ETH_adj, dtype: float64

In [44]:
total_weth_value = round(hsds.main(chosen_group), 6)
total_weth_value

0.485128

In [70]:

import importlib
import helen0822 as hsds

importlib.reload(hsds)

df = df_cleaned.copy()

for chosen_hash in hashes:

    chosen_group = df_grouped.get_group(chosen_hash)

    source_node = hsds.find_source_node(chosen_group)

    chosen_group, source_node = hsds.hashes_to_letters(chosen_group, source_node)

    total_weth_value = hsds.main(chosen_group)

    #######################################################################################################################################
    if total_weth_value is None:
        total_weth_value = -1 # untraceable transactions
    else:
        try:
            total_weth_value = round(total_weth_value, 6)
        except (TypeError, ValueError) as e:
            print(f"When processing the hash ", chosen_hash)
            print(f"An error occurred: {e}") #If this never gets printed, update the code for the one below!
            total_weth_value = -2
    #######################################################################################################################################
    # if total_weth_value is None:
    #     total_weth_value = -1 # untraceable transactions      
    # else:
    #     total_weth_value = round(total_weth_value, 6)
    #######################################################################################################################################

    # Adding an extra column on df_cleaned for our guess of Transaction_Value:
    # Use .loc to update the DataFrame
    df.loc[df['hash'] == chosen_hash, 'Transaction_Value'] = total_weth_value
    # df[df['hash'] == chosen_hash]['Transaction_Value'] = total_weth_value

    # We can try to guess the fee as well:
    # df_cleaned[df_cleaned['hash'] == chosen_hash]['Transaction_Fee_'] = valuemin

In [71]:
df.head()

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
980,0x4488cd3c8cce4047880d6b6383df0696f961fd994eb8...,internal,0xf43e889444e95a0429c32b0b601dc16edf90fdbf,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0.0,ETH,staticcall,swap,0.160000,NaN,0.160000
981,0x320291b8f33ae570cf169d40512dfd8256bcf47273f1...,internal,0xf7a2f863299c17dfa11cd8a14e7c7dca92f315b9,0xa41161af8d4ee421ba5fed4328f2b12034796a21,0.0,ETH,staticcall,swap,0.360602,NaN,-1.000000
982,0xbc891d9fa036b11150532abf897b38ba00d1b2a0c503...,erc_full,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.1,WETH,ERC20,swap,0.100000,NaN,0.100000
983,0xd4af7a2137d217b959dc6e154467885e01cc013033de...,main,0x8ff7d1634c81ae41bc2fba4854c166dab50887b3,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.0,ETH,main,swap,0.050422,NaN,0.050422
984,0xfa16cd33441ab64a9699cf879fecc05229a3ec80f480...,internal,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0.0,ETH,staticcall,swap,0.117748,NaN,0.117748


In [72]:
# Save DataFrame to Parquet file
df.to_parquet('./data/processed_df.pqt')

In [1]:
##for tomorrow:
import pandas as pd

# Read the DataFrame from the Parquet file
new_df = pd.read_parquet('./data/processed_df.pqt')

# Display the DataFrame
new_df.head()

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
980,0x4488cd3c8cce4047880d6b6383df0696f961fd994eb8...,internal,0xf43e889444e95a0429c32b0b601dc16edf90fdbf,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0.0,ETH,staticcall,swap,0.160000,NaN,0.160000
981,0x320291b8f33ae570cf169d40512dfd8256bcf47273f1...,internal,0xf7a2f863299c17dfa11cd8a14e7c7dca92f315b9,0xa41161af8d4ee421ba5fed4328f2b12034796a21,0.0,ETH,staticcall,swap,0.360602,NaN,-1.000000
982,0xbc891d9fa036b11150532abf897b38ba00d1b2a0c503...,erc_full,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.1,WETH,ERC20,swap,0.100000,NaN,0.100000
983,0xd4af7a2137d217b959dc6e154467885e01cc013033de...,main,0x8ff7d1634c81ae41bc2fba4854c166dab50887b3,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.0,ETH,main,swap,0.050422,NaN,0.050422
984,0xfa16cd33441ab64a9699cf879fecc05229a3ec80f480...,internal,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0.0,ETH,staticcall,swap,0.117748,NaN,0.117748


In [74]:
new_df.columns

Index(['hash', 'source', 'from', 'to', 'value', 'symbol', 'sub_type', 'type',
       'ETH_adj', 'transaction_fee', 'Transaction_Value'],
      dtype='object')

In [75]:
type(new_df)

pandas.core.frame.DataFrame

In [3]:
def is_match(group):
    return (group['ETH_adj'] == group['Transaction_Value']).all()

# Group by 'hash' and apply the match function
match_results = new_df.groupby('hash').apply(is_match).reset_index(name='match')

# Calculate the percentage of hashes that match
percentage_matches = (match_results['match'].mean()) * 100

# Display the results
print(f"Percentage of hashes where all values match: {percentage_matches:.2f}%")
# Percentage of hashes where all values match: 12.68%


Percentage of hashes where all values match: 12.68%


In [77]:
import importlib
import helen0822 as hsds

importlib.reload(hsds)

df2 = df_cleaned.copy()

for chosen_hash in hashes:

    chosen_group = df_grouped.get_group(chosen_hash)

    source_node = hsds.find_source_node(chosen_group)

    chosen_group, source_node = hsds.hashes_to_letters(chosen_group, source_node)

    G, fig = hsds.draw_base_graph_four(chosen_group, source_node, create_using=nx.MultiDiGraph)

    if hsds.count_connected_components(G, mode='weak') == 1:

        total_weth_value = hsds.main(chosen_group)

        if total_weth_value is None:
            total_weth_value = -1 # untraceable transactions      
        else:
            total_weth_value = round(total_weth_value, 6)
    else:
        total_weth_value = -2
        
    df2.loc[df['hash'] == chosen_hash, 'Transaction_Value'] = total_weth_value

df2.to_parquet('./data/processed_df2.pqt')

#################################################################################################################################
new_df2 = pd.read_parquet('./data/processed_df2.pqt')

def is_match(group):
    return (group['ETH_adj'] == group['Transaction_Value']).all()

# Group by 'hash' and apply the match function
match_results = new_df2.groupby('hash').apply(is_match).reset_index(name='match')

# Calculate the percentage of hashes that match
percentage_matches = (match_results['match'].mean()) * 100

# Display the results
print(f"Percentage of hashes where all values match: {percentage_matches:.2f}%")
# Percentage of hashes where all values match: it should be the same, I should filter for Transacation_value > 0 ?

Percentage of hashes where all values match: 12.68%


In [ ]:
#####################################################################################################################################################3

<!-- TASKS:

Run the above algorithm
Analyse the data
Add more data!!
Repeat
Generate tables and or graphs to incorporate in the report -->

In [78]:
new_df.columns

Index(['hash', 'source', 'from', 'to', 'value', 'symbol', 'sub_type', 'type',
       'ETH_adj', 'transaction_fee', 'Transaction_Value'],
      dtype='object')

In [28]:
df_main = new_df[new_df['source'] == 'main']

# Drop duplicates to keep only the first occurrence of each hash
df_unique_hashes = df_main.drop_duplicates(subset='hash')

In [29]:
df_unique_hashes.head()

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
983,0xd4af7a2137d217b959dc6e154467885e01cc013033de...,main,0x8ff7d1634c81ae41bc2fba4854c166dab50887b3,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.0,ETH,main,swap,0.050422,NaN,0.050422
1025,0x69a3c703f66dc1dd0a58540853c2c7ffbcab7a19c63a...,main,0xcc202930867769a83b61cf5053b65d1845e76aea,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.0,ETH,main,swap,4.203805,NaN,-1.000000
1028,0x9ee9c78dc615794599460906e1a56fc938313ee32a1f...,main,0xfe24a602a28e0dc67a02c689888a42566e77e2f4,0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45,0.0,ETH,main,swap,0.105325,NaN,-1.000000
1040,0x032c5e405f0c906692da3c862747c8058e89791b0509...,main,0xece51b7d6b63b7f5a976ca0604dff6605f67bddb,0x7a250d5630b4cf539739df2c5dacb4c659f2488d,0.0,ETH,main,remove liquitidy: removeLiquidityETHWithPermit...,0.414086,NaN,-1.000000
1051,0xdfff22fbf30787c8eb993beb0feed154b2c536293b6d...,main,0xd3738e027c271462031b41653e47734e54a4beab,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.5,ETH,main,swap,0.500000,NaN,0.500000


In [30]:
len(df_unique_hashes)
df_unique_hashes

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
983,0xd4af7a2137d217b959dc6e154467885e01cc013033de...,main,0x8ff7d1634c81ae41bc2fba4854c166dab50887b3,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.000000,ETH,main,swap,0.050422,NaN,0.050422
1025,0x69a3c703f66dc1dd0a58540853c2c7ffbcab7a19c63a...,main,0xcc202930867769a83b61cf5053b65d1845e76aea,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.000000,ETH,main,swap,4.203805,NaN,-1.000000
1028,0x9ee9c78dc615794599460906e1a56fc938313ee32a1f...,main,0xfe24a602a28e0dc67a02c689888a42566e77e2f4,0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45,0.000000,ETH,main,swap,0.105325,NaN,-1.000000
1040,0x032c5e405f0c906692da3c862747c8058e89791b0509...,main,0xece51b7d6b63b7f5a976ca0604dff6605f67bddb,0x7a250d5630b4cf539739df2c5dacb4c659f2488d,0.000000,ETH,main,remove liquitidy: removeLiquidityETHWithPermit...,0.414086,NaN,-1.000000
1051,0xdfff22fbf30787c8eb993beb0feed154b2c536293b6d...,main,0xd3738e027c271462031b41653e47734e54a4beab,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.500000,ETH,main,swap,0.500000,NaN,0.500000
...,...,...,...,...,...,...,...,...,...,...,...
1670919,0x48bca705f223c71680a31f68186d63e96c61386459bc...,main,0x267d22b609b74c4e5498a49db403fcb610cc94ce,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.100000,ETH,main,swap,0.100000,NaN,0.200000
1671011,0x1e471a5e47a1608ae4cc02d1ea67475439d77347e465...,main,0x91b91584c455bfeadf78cb695b86dce187137fb6,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,2.000000,ETH,main,swap,2.000000,NaN,4.000000
1671025,0x9d585623ea5ed03788bf62f14b88190102b9a89e5319...,main,0x9d2a4b27f4b649a1cb542ec1fe9497dc8d2e3bd9,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.000000,ETH,main,swap,1.908960,NaN,-1.000000
1671034,0x202a77f3136a70ef010dc08e3a90b7e3a24d077acb0e...,main,0xafb919f6f1ca8a1b2a6ac52c15dd28e4ab319375,0x1111111254eeb25477b68fb85ed929f73a960582,0.000000,ETH,main,swap,0.019659,NaN,0.019659


In [83]:
def calculate_match_percentage(df):
    # Filter rows where Transaction_Value > 0
    df_filtered = df[df['Transaction_Value'] > 0]

    matches = (df_filtered['Transaction_Value'] == df_filtered['ETH_adj']).sum()
    total = len(df_filtered)

    percentage = (matches / total) * 100

    return percentage

percentage_matches = calculate_match_percentage(df_unique_hashes)
print(f"Percentage of matches where Transaction_Value > 0: {percentage_matches:.2f}%")

Percentage of matches where Transaction_Value > 0: 13.89%


In [91]:
import polars as pl

def calculate_negative_one_percentage(df):
    # Count the number of -1 values in 'Transaction_Value'
    count_negative_one = (df['Transaction_Value'] == -1).sum()
    total_rows = len(df)

    percentage = (count_negative_one / total_rows) * 100

    return percentage

# Calculate and print the percentage of -1 values
percentage_negative_one = calculate_negative_one_percentage(df_unique_hashes)
print(f"Percentage of Transaction_Value that is -1: {percentage_negative_one:.2f}%")

Percentage of Transaction_Value that is -1: 7.71%


In [6]:
df_unique_hashes.head()


,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
983,0xd4af7a2137d217b959dc6e154467885e01cc013033de...,main,0x8ff7d1634c81ae41bc2fba4854c166dab50887b3,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.0,ETH,main,swap,0.050422,NaN,0.050422
1025,0x69a3c703f66dc1dd0a58540853c2c7ffbcab7a19c63a...,main,0xcc202930867769a83b61cf5053b65d1845e76aea,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.0,ETH,main,swap,4.203805,NaN,-1.000000
1028,0x9ee9c78dc615794599460906e1a56fc938313ee32a1f...,main,0xfe24a602a28e0dc67a02c689888a42566e77e2f4,0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45,0.0,ETH,main,swap,0.105325,NaN,-1.000000
1040,0x032c5e405f0c906692da3c862747c8058e89791b0509...,main,0xece51b7d6b63b7f5a976ca0604dff6605f67bddb,0x7a250d5630b4cf539739df2c5dacb4c659f2488d,0.0,ETH,main,remove liquitidy: removeLiquidityETHWithPermit...,0.414086,NaN,-1.000000
1051,0xdfff22fbf30787c8eb993beb0feed154b2c536293b6d...,main,0xd3738e027c271462031b41653e47734e54a4beab,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.5,ETH,main,swap,0.500000,NaN,0.500000


In [105]:
df_unique_hashes.head(12)

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
983,0xd4af7a2137d217b959dc6e154467885e01cc013033de...,main,0x8ff7d1634c81ae41bc2fba4854c166dab50887b3,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.00,ETH,main,swap,0.050422,NaN,0.050422
1025,0x69a3c703f66dc1dd0a58540853c2c7ffbcab7a19c63a...,main,0xcc202930867769a83b61cf5053b65d1845e76aea,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.00,ETH,main,swap,4.203805,NaN,-1.000000
1028,0x9ee9c78dc615794599460906e1a56fc938313ee32a1f...,main,0xfe24a602a28e0dc67a02c689888a42566e77e2f4,0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45,0.00,ETH,main,swap,0.105325,NaN,-1.000000
1040,0x032c5e405f0c906692da3c862747c8058e89791b0509...,main,0xece51b7d6b63b7f5a976ca0604dff6605f67bddb,0x7a250d5630b4cf539739df2c5dacb4c659f2488d,0.00,ETH,main,remove liquitidy: removeLiquidityETHWithPermit...,0.414086,NaN,-1.000000
1051,0xdfff22fbf30787c8eb993beb0feed154b2c536293b6d...,main,0xd3738e027c271462031b41653e47734e54a4beab,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.50,ETH,main,swap,0.500000,NaN,0.500000
1104,0x0f895c9b4bc9f551f8be1fad09b13511a7e41f3ddd0a...,main,0x07d3b6ab4870fcbb639dd224b449fbd6927e1408,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.00,ETH,main,swap,0.136165,NaN,-1.000000
1108,0x017d56e10ef02ddd20f2addfa4100d0c17d3c007ed86...,main,0xe67d2f74e9abf55671de5f5acfb3fc39248c171a,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,2.30,ETH,main,swap,2.300000,NaN,4.600000
1113,0x3f0f91f314613e0ddfd8c5e92da0046957c171790ae8...,main,0x4116757395c3864a99cf733e4884f44948e27466,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.00,ETH,main,swap,0.124909,NaN,0.124909
1123,0xc8ef407da3a49928b9d6f3f4d82340803c397030ef48...,main,0x2793abd233727eef86bd8ccdd2dc57c43377325e,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.05,ETH,main,swap,0.050000,NaN,0.050000
1124,0xe13e7f1866befa4d25c1504b313b3031467bbac91196...,main,0xb6137b8a2b4577568a19234722034889beb5b574,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.50,ETH,main,swap,0.500000,NaN,1.000000


In [ ]:
#####################################################################################################################################################3

In [93]:
def calculate_match_percentage(df):
    # Filter rows where Transaction_Value > 0
    df_filtered = df[df['Transaction_Value'] > 0]

    matches = (df_filtered['Transaction_Value'] == df_filtered['ETH_adj']).sum()
    total = len(df_filtered)

    percentage = (matches / total) * 100

    return percentage

percentage_matches = calculate_match_percentage(df_unique_hashes)
print(f"Percentage of matches where Transaction_Value > 0: {percentage_matches:.2f}%")

Percentage of matches where Transaction_Value > 0: 16.28%


In [ ]:
df.head()

In [101]:
def calculate_match_percentage_per_type(df):
    df_filtered = df[df['Transaction_Value'] > 0]

    grouped = df_filtered.groupby('type')

    # Define a function to calculate match percentage within each group
    def group_match_percentage(group):
        matches = (group['Transaction_Value'] == group['ETH_adj']).sum()
        total = len(group)
        if total > 0:
            return (matches / total) * 100
        else:
            return 0

    match_percentages = grouped.apply(group_match_percentage).reset_index(name='Match_Percentage')

    # Ensure all unique types are included even if they have no matches
    all_types = df['type'].unique()
    match_percentages = pd.DataFrame({
        'type': all_types
    }).merge(match_percentages, on='type', how='left').fillna(0)

    return match_percentages

match_percentages = calculate_match_percentage_per_type(df_unique_hashes)
print(match_percentages)

                                                 type  Match_Percentage
0                                                swap         13.918274
1   remove liquitidy: removeLiquidityETHWithPermit...          0.000000
2                                             execute          0.000000
3                                            transfer          0.000000
4   swapExactTokensForETHSupportingFeeOnTransferTo...          0.000000
5                                          non traité          0.000000
6         remove liquitidy: removeLiquidityWithPermit          0.000000
7                                          0x8873fca9          0.000000
8                                      withdrawTokens          0.000000
9                   remove liquitidy: addLiquidityETH          0.000000
10                     add liquidity: addLiquidityETH          0.000000
11                                       transferFrom          0.000000
12                                    harvestMultiple          0

In [97]:
num_unique_types = df_unique_hashes['type'].nunique()

print(f"Number of unique values in the 'type' column: {num_unique_types}")

Number of unique values in the 'type' column: 46


In [98]:
df_unique_hashes['type'].unique()

array(['swap',
       'remove liquitidy: removeLiquidityETHWithPermitSupportingFeeOnTransferTokens',
       'execute', 'transfer',
       'swapExactTokensForETHSupportingFeeOnTransferTokens', 'non traité',
       'remove liquitidy: removeLiquidityWithPermit', '0x8873fca9',
       'withdrawTokens', 'remove liquitidy: addLiquidityETH',
       'add liquidity: addLiquidityETH', 'transferFrom',
       'harvestMultiple', 'harvest', 'claimChoice',
       'add liquidity: multicall', 'remove liquitidy: multicall',
       '0x3d21e25a', 'remove liquitidy: 0x83f31917', '0xb1c191e2',
       'bridge', 'add liquidity: mint', '0x2aac3cac', '0xff39799a',
       '0x3a3f7332', 'remove liquitidy: collect', '0xff7542d1',
       'increaseLiquidityOptimal', 'remove liquitidy: 0x90cc74ac',
       'remove liquitidy: 0x3070c305', 'remove liquitidy: 0x3d21e25a',
       'unoswap', 'multicall',
       'remove liquitidy: removeLiquidityETHWithPermit', 'swapSimpleMode',
       '0xd9627aa4', 'remove liquitidy: 0x0a62

In [99]:
type_counts = df_unique_hashes['type'].value_counts()

# Convert to DataFrame for better readability
type_counts_df = type_counts.reset_index()
type_counts_df.columns = ['type', 'Count']

# Print the DataFrame
print(type_counts_df)

                                                 type  Count
0                                                swap  72770
1                         remove liquitidy: multicall    200
2                            add liquidity: multicall     83
3                                             execute     59
4                           remove liquitidy: collect     45
5                                            transfer     28
6                                          non traité     18
7                                          0x70fef1da     18
8   swapExactTokensForETHSupportingFeeOnTransferTo...     17
9                      add liquidity: addLiquidityETH     15
10                                             bridge     12
11                                add liquidity: mint      9
12                                         0x554db991      6
13                                         0x3d21e25a      6
14                                         0xff7542d1      5
15                      

In [106]:
import helen0822 as hsds

chonse_hash = "0xe67d2f74e9abf55671de5f5acfb3fc39248c171a"
chosen_df = df_unique_hashes[df_unique_hashes['hash'] == chosen_hash]

chosen_df

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
1626559,0x3a92d4b66517d58ec170f614a4f72281e679e4edd124...,main,0xb80d90fcf2ed0e4febe02d2a209109bf1f62df95,0x2626664c2603336e57b271c5c0b26f421741e481,0.0,ETH,main,swap,0.102581,NaN,0.102581


In [107]:
df_unique_hashes.head(12)

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
983,0xd4af7a2137d217b959dc6e154467885e01cc013033de...,main,0x8ff7d1634c81ae41bc2fba4854c166dab50887b3,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.00,ETH,main,swap,0.050422,NaN,0.050422
1025,0x69a3c703f66dc1dd0a58540853c2c7ffbcab7a19c63a...,main,0xcc202930867769a83b61cf5053b65d1845e76aea,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.00,ETH,main,swap,4.203805,NaN,-1.000000
1028,0x9ee9c78dc615794599460906e1a56fc938313ee32a1f...,main,0xfe24a602a28e0dc67a02c689888a42566e77e2f4,0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45,0.00,ETH,main,swap,0.105325,NaN,-1.000000
1040,0x032c5e405f0c906692da3c862747c8058e89791b0509...,main,0xece51b7d6b63b7f5a976ca0604dff6605f67bddb,0x7a250d5630b4cf539739df2c5dacb4c659f2488d,0.00,ETH,main,remove liquitidy: removeLiquidityETHWithPermit...,0.414086,NaN,-1.000000
1051,0xdfff22fbf30787c8eb993beb0feed154b2c536293b6d...,main,0xd3738e027c271462031b41653e47734e54a4beab,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.50,ETH,main,swap,0.500000,NaN,0.500000
1104,0x0f895c9b4bc9f551f8be1fad09b13511a7e41f3ddd0a...,main,0x07d3b6ab4870fcbb639dd224b449fbd6927e1408,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.00,ETH,main,swap,0.136165,NaN,-1.000000
1108,0x017d56e10ef02ddd20f2addfa4100d0c17d3c007ed86...,main,0xe67d2f74e9abf55671de5f5acfb3fc39248c171a,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,2.30,ETH,main,swap,2.300000,NaN,4.600000
1113,0x3f0f91f314613e0ddfd8c5e92da0046957c171790ae8...,main,0x4116757395c3864a99cf733e4884f44948e27466,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.00,ETH,main,swap,0.124909,NaN,0.124909
1123,0xc8ef407da3a49928b9d6f3f4d82340803c397030ef48...,main,0x2793abd233727eef86bd8ccdd2dc57c43377325e,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.05,ETH,main,swap,0.050000,NaN,0.050000
1124,0xe13e7f1866befa4d25c1504b313b3031467bbac91196...,main,0xb6137b8a2b4577568a19234722034889beb5b574,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.50,ETH,main,swap,0.500000,NaN,1.000000


In [7]:
df_different = df_unique_hashes[df_unique_hashes['ETH_adj'] != df_unique_hashes['Transaction_Value'] ]

df_same = df_unique_hashes[df_unique_hashes['ETH_adj'] == df_unique_hashes['Transaction_Value'] ]

In [8]:
len(df_different) + len(df_same)

73341

In [110]:
len(df_same)

9302

In [111]:
df_different.head()

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
983,0xd4af7a2137d217b959dc6e154467885e01cc013033de...,main,0x8ff7d1634c81ae41bc2fba4854c166dab50887b3,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.0,ETH,main,swap,0.050422,NaN,0.050422
1025,0x69a3c703f66dc1dd0a58540853c2c7ffbcab7a19c63a...,main,0xcc202930867769a83b61cf5053b65d1845e76aea,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.0,ETH,main,swap,4.203805,NaN,-1.000000
1028,0x9ee9c78dc615794599460906e1a56fc938313ee32a1f...,main,0xfe24a602a28e0dc67a02c689888a42566e77e2f4,0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45,0.0,ETH,main,swap,0.105325,NaN,-1.000000
1040,0x032c5e405f0c906692da3c862747c8058e89791b0509...,main,0xece51b7d6b63b7f5a976ca0604dff6605f67bddb,0x7a250d5630b4cf539739df2c5dacb4c659f2488d,0.0,ETH,main,remove liquitidy: removeLiquidityETHWithPermit...,0.414086,NaN,-1.000000
1104,0x0f895c9b4bc9f551f8be1fad09b13511a7e41f3ddd0a...,main,0x07d3b6ab4870fcbb639dd224b449fbd6927e1408,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.0,ETH,main,swap,0.136165,NaN,-1.000000


In [112]:
type(df_unique_hashes["ETH_adj"])

pandas.core.series.Series

In [64]:
import pandas as pd

def calculate_match_percentage_per_type(df, tolerance=1e-6):
    # Filter rows where Transaction_Value > 0
    df_filtered = df[df['Transaction_Value'] > 0]
    # df_filtered = df.copy()

    grouped = df_filtered.groupby('type')

    # Define a function to calculate match percentage within each group
    def group_match_percentage(group):
        # Check if values are close within the specified tolerance
        matches = ((abs(group['Transaction_Value'] - group['ETH_adj']) < tolerance)).sum()
        total = len(group)
        if total > 0:
            return (matches / total) * 100
        else:
            return 0

    # Apply the function to each group and reset index to get a DataFrame
    match_percentages = grouped.apply(group_match_percentage).reset_index(name='Match_Percentage')

    # Ensure all unique types are included even if they have no matches
    all_types = df['type'].unique()
    match_percentages = pd.DataFrame({
        'type': all_types
    }).merge(match_percentages, on='type', how='left').fillna(0)

    return match_percentages

# Calculate and print the match percentages per group
match_percentages = calculate_match_percentage_per_type(df)
print(match_percentages)


                                                 type  Match_Percentage
0                                                swap         59.452666
1   remove liquitidy: removeLiquidityETHWithPermit...          0.000000
2                                             execute         90.000000
3                                            transfer          0.000000
4   swapExactTokensForETHSupportingFeeOnTransferTo...         61.538462
5                                          non traité         66.666667
6         remove liquitidy: removeLiquidityWithPermit          0.000000
7                                          0x8873fca9          0.000000
8                                      withdrawTokens        100.000000
9                   remove liquitidy: addLiquidityETH          0.000000
10                     add liquidity: addLiquidityETH          0.000000
11                                       transferFrom          0.000000
12                                    harvestMultiple          0

In [10]:
import pandas as pd

def calculate_overall_match_percentage(df, tolerance=0.0005):
    # Filter rows where Transaction_Value > 0
    df_filtered = df[df['Transaction_Value'] > 0]

    matches = (abs(df_filtered['Transaction_Value'] - df_filtered['ETH_adj']) < tolerance).sum()
    total = len(df_filtered)

    if total > 0:
        percentage = (matches / total) * 100
    else:
        percentage = 0

    return percentage

# Calculate and print the overall match percentage
overall_percentage = calculate_overall_match_percentage(df_unique_hashes)
print(f"Overall match percentage: {overall_percentage:.2f}%")

Overall match percentage: 61.47%


In [11]:

df_unique_hashes_filtered = df_unique_hashes[df_unique_hashes['Transaction_Value'] > 0]

df_different = df_unique_hashes_filtered[df_unique_hashes_filtered['ETH_adj'] - df_unique_hashes_filtered['Transaction_Value'] > 0.0005]

df_same = df_unique_hashes_filtered[df_unique_hashes_filtered['ETH_adj'] - df_unique_hashes_filtered['Transaction_Value'] < 0.0005]

In [133]:
df_different.head(12)

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value
3697,0x4531a62cb950c9487c456eac66c118baa53dd4573c54...,main,0x41f8bf5ea23c42a03c172c74317ef8bd9946d5cc,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.165920,ETH,main,swap,0.165920,NaN,0.157270
6636,0x0517891a9a32b263e2e72fff57d3767bad366168439a...,main,0xd9bf38fc865e5a64aa7927cb9f5f0b299ac5b0b7,0x2ec705d306b51e486b1bc0d6ebee708e0661add1,0.129964,ETH,main,swap,0.129964,NaN,0.128664
7881,0x2c44f83d40a023d66ef8635792484e30b8a17e2ecdf4...,main,0x13da4f6b997c046e8478f60d144ac2ba1325a1c3,0xbe6fee3756f7be3a0cd492059341cb5b77dd81f9,0.210000,ETH,main,swap,0.210000,NaN,0.208950
9624,0x11944684d8971977887d3a512177de4a46ff80c7e0c2...,main,0x9cdd9650eef80a2a873be9b0fe7dc0473c006b08,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.569034,ETH,main,swap,0.569034,NaN,0.561972
13136,0x9e04f721a083c1e191fc97adb48bbb6417cca68fe612...,main,0x96adfc3ca77380f38f90f1bd4d6e6fe3fd82f722,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.266738,ETH,main,swap,0.266738,NaN,0.265411
14694,0xdc09770ddcc4e96fb4e0fab83cdd941706308f500fe3...,main,0x96adfc3ca77380f38f90f1bd4d6e6fe3fd82f722,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.518468,ETH,main,swap,0.518468,NaN,0.510431
14948,0x0dc551bdb14d7791ddafc88707e355ebccb318abb14e...,main,0x9f7080d6f639e3e74183dc6cf408cf00017c30a5,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.064259,ETH,main,swap,0.064259,NaN,0.061199
16803,0x662d7867be889257ab95f9d19896d054023958195925...,main,0x30998e68e9f2a532131f69811fbb88870aa0389f,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.006249,ETH,main,swap,0.006249,NaN,0.004999
17621,0x059f58a599f720bc8318810bf49a7ff5177465cec377...,main,0xde5f51eacb8098a07ebe9a8086d008c4859dcca9,0xef1c6e67703c7bd7107eed8303fbe6ec2554bf6b,0.016882,ETH,main,swap,0.016882,NaN,0.015927
20659,0xc7f7208060737cc8bd099ada24a16301d5baa6c598db...,main,0x77d5737d5c44d811ee8a099ba664a7d250c07040,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0.320048,ETH,main,swap,0.320048,NaN,0.301932


In [24]:
import pandas as pd
import numpy as np

def calculate_overall_match_percentage(df, percentage_tolerance=1.0):
    # Convert percentage tolerance to absolute tolerance
    tolerance = percentage_tolerance / 100

    # Filter rows where Transaction_Value > 0
    df_filtered = df[df['Transaction_Value'] > 0]

    # Drop rows with NaN values in 'Transaction_Value' or 'ETH_adj'
    df_filtered = df_filtered.dropna(subset=['Transaction_Value', 'ETH_adj'])

    # Calculate tolerance in absolute terms
    df_filtered['Tolerance'] = df_filtered['ETH_adj'] * tolerance

    # Check if values are close within the specified tolerance
    matches = (abs(df_filtered['Transaction_Value'] - df_filtered['ETH_adj']) <= df_filtered['Tolerance']).sum()
    total = len(df_filtered)

    if total > 0:
        percentage = (matches / total) * 100
    else:
        percentage = 0

    return percentage

# Calculate and print the overall match percentage with a 1% tolerance
overall_percentage = calculate_overall_match_percentage(df_unique_hashes_filtered, percentage_tolerance=1.0)
print(f"Overall match percentage with 1% tolerance: {overall_percentage:.2f}%")

Overall match percentage with 1% tolerance: 61.42%


In [25]:
def apply_tolerance(df, percentage_tolerance=1.0):
    # Filter rows where Transaction_Value > 0
    df_filtered = df[df['Transaction_Value'] > 0].copy()

    # Calculate tolerance in absolute terms for each row
    tolerance = df_filtered['ETH_adj'] * (percentage_tolerance / 100)

    # Create a column for tolerance and check the absolute difference
    df_filtered['Tolerance'] = tolerance
    df_filtered['Difference'] = abs(df_filtered['ETH_adj'] - df_filtered['Transaction_Value'])
    
    # Define matching and non-matching rows based on tolerance
    df_different = df_filtered[df_filtered['Difference'] > df_filtered['Tolerance']]
    df_same = df_filtered[df_filtered['Difference'] <= df_filtered['Tolerance']]

    return df_different, df_same

# Calculate and print the DataFrames with a 1% tolerance
df_different, df_same = apply_tolerance(df_unique_hashes, percentage_tolerance=1.0)

print("DataFrame with Differences Exceeding Tolerance:")
print(len(df_different))

print("\nDataFrame with Matches Within Tolerance:")
print(len(df_same))

DataFrame with Differences Exceeding Tolerance:
25844

DataFrame with Matches Within Tolerance:
41142


In [26]:
len(df_same) / (len(df_different) + len(df_same))

0.6141880392917923

In [32]:
df = df_unique_hashes.copy()
df_grouped = df.groupby('hash')
hashes = df_unique_hashes['hash']

import helen0822 as hsds
import networkx as nx

for chosen_hash in hashes:

    chosen_group = df_grouped.get_group(chosen_hash)

    source_node = hsds.find_source_node(chosen_group)

    G, fig = hsds.draw_base_graph_four(chosen_group, source_node, create_using=nx.MultiDiGraph)

    df['Connected_Components'] = hsds.count_connected_components(G, mode='weak')


In [59]:
df.to_parquet('./data/processed_df12.pqt')

df_12 = pd.read_parquet('./data/processed_df12.pqt')

In [2]:
import pandas as pd
df12 = pd.read_parquet('./data/processed_df12.pqt')

In [3]:
df12[df12['hash'] == "0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7a1a6c7741fcfd6db0172"]

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value,Connected_Components
768911,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,main,0x7a933231fbc3ebcf1a2eab47447e87568870e9c1,0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45,0.0,ETH,main,non traité,0.06112,NaN,-1.0,1


In [4]:

df_final = pd.read_parquet('./data/final_result.pqt')

df_final_grouped = df_final.groupby('hash')
hashes = df_final_grouped['hash']

import helen0822 as hsds
import networkx as nx

chosen_hash = "0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7a1a6c7741fcfd6db0172"

chosen_group = df_final_grouped.get_group(chosen_hash)

source_node = hsds.find_source_node(chosen_group)

G, fig = hsds.draw_base_graph_four(chosen_group, source_node, create_using=nx.MultiDiGraph)

fig

In [5]:
chosen_group

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee
1283446,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc,0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758,0x7a933231fbc3ebcf1a2eab47447e87568870e9c1,2.619030e+04,ECT,ERC20,non traité,0.06112,NaN
1283447,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc,0xcb6ae57a26080a518348f9aac01a0e82fa6818f1,0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758,6.111984e-02,WETH,ERC20,non traité,0.06112,NaN
1283448,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc_full,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x08a1059E17ce0bbC3E966F4AB7a0e0a765E2805E,2.704797e+03,SHIB,ERC20,non traité,0.06112,NaN
1283449,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc_full,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x34D4b50d6244145fbe229A21aB1155CEFd506Eee,1.869746e+03,SHIB,ERC20,non traité,0.06112,NaN
1283450,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc_full,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x51a38fdf5FAc32EEA44a593B209c0AB2B67CdCd9,3.376653e+03,SHIB,ERC20,non traité,0.06112,NaN
1283451,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc_full,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x522583Bf0E7d92D7D4Faa3739Cf858C5a8F54210,2.339060e+03,SHIB,ERC20,non traité,0.06112,NaN
1283452,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc_full,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x60A01B3E0D9D6732B3031F12eaD171Cb5cfdc949,5.168041e+02,SHIB,ERC20,non traité,0.06112,NaN
1283453,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc_full,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x657C4Df69874B1a1613a8591809A0560De1ca92F,6.842444e+02,SHIB,ERC20,non traité,0.06112,NaN
1283454,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc_full,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x7a933231FbC3eBcf1a2eAB47447E87568870e9C1,1.746554e+04,SHIB,ERC20,non traité,0.06112,NaN
1283455,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,erc_full,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x95Ebfe8624D024711dfa869980D84174a1ACb046,1.667273e+03,SHIB,ERC20,non traité,0.06112,NaN


In [6]:
type(chosen_group)

pandas.core.frame.DataFrame

In [7]:
chosen_group.columns

Index(['hash', 'source', 'from', 'to', 'value', 'symbol', 'sub_type', 'type',
       'ETH_adj', 'transaction_fee'],
      dtype='object')

In [8]:
columns = ['hash', 'from', 'to', 'value', 'symbol', 'sub_type', 'type',
       'ETH_adj', 'transaction_fee']

filtered_chosen_group = chosen_group[columns].copy()

In [9]:
filtered_chosen_group.drop_duplicates()

,hash,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee
1283446,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758,0x7a933231fbc3ebcf1a2eab47447e87568870e9c1,2.619030e+04,ECT,ERC20,non traité,0.06112,NaN
1283447,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0xcb6ae57a26080a518348f9aac01a0e82fa6818f1,0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758,6.111984e-02,WETH,ERC20,non traité,0.06112,NaN
1283448,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x08a1059E17ce0bbC3E966F4AB7a0e0a765E2805E,2.704797e+03,SHIB,ERC20,non traité,0.06112,NaN
1283449,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x34D4b50d6244145fbe229A21aB1155CEFd506Eee,1.869746e+03,SHIB,ERC20,non traité,0.06112,NaN
1283450,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x51a38fdf5FAc32EEA44a593B209c0AB2B67CdCd9,3.376653e+03,SHIB,ERC20,non traité,0.06112,NaN
1283451,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x522583Bf0E7d92D7D4Faa3739Cf858C5a8F54210,2.339060e+03,SHIB,ERC20,non traité,0.06112,NaN
1283452,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x60A01B3E0D9D6732B3031F12eaD171Cb5cfdc949,5.168041e+02,SHIB,ERC20,non traité,0.06112,NaN
1283453,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x657C4Df69874B1a1613a8591809A0560De1ca92F,6.842444e+02,SHIB,ERC20,non traité,0.06112,NaN
1283454,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x7a933231FbC3eBcf1a2eAB47447E87568870e9C1,1.746554e+04,SHIB,ERC20,non traité,0.06112,NaN
1283455,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x95Ebfe8624D024711dfa869980D84174a1ACb046,1.667273e+03,SHIB,ERC20,non traité,0.06112,NaN


In [10]:
filtered_chosen_group = filtered_chosen_group[filtered_chosen_group['value'] != 0]

In [11]:
filtered_chosen_group

,hash,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee
1283446,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758,0x7a933231fbc3ebcf1a2eab47447e87568870e9c1,2.619030e+04,ECT,ERC20,non traité,0.06112,NaN
1283447,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0xcb6ae57a26080a518348f9aac01a0e82fa6818f1,0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758,6.111984e-02,WETH,ERC20,non traité,0.06112,NaN
1283448,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x08a1059E17ce0bbC3E966F4AB7a0e0a765E2805E,2.704797e+03,SHIB,ERC20,non traité,0.06112,NaN
1283449,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x34D4b50d6244145fbe229A21aB1155CEFd506Eee,1.869746e+03,SHIB,ERC20,non traité,0.06112,NaN
1283450,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x51a38fdf5FAc32EEA44a593B209c0AB2B67CdCd9,3.376653e+03,SHIB,ERC20,non traité,0.06112,NaN
1283451,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x522583Bf0E7d92D7D4Faa3739Cf858C5a8F54210,2.339060e+03,SHIB,ERC20,non traité,0.06112,NaN
1283452,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x60A01B3E0D9D6732B3031F12eaD171Cb5cfdc949,5.168041e+02,SHIB,ERC20,non traité,0.06112,NaN
1283453,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x657C4Df69874B1a1613a8591809A0560De1ca92F,6.842444e+02,SHIB,ERC20,non traité,0.06112,NaN
1283454,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x7a933231FbC3eBcf1a2eAB47447E87568870e9C1,1.746554e+04,SHIB,ERC20,non traité,0.06112,NaN
1283455,0xf5d1ecb8d73b60851ffc17080146fb188f6ceb3c95d7...,0x515BF7af2ff2e5Def9FFd9922Dd34041BE3D1b79,0x95Ebfe8624D024711dfa869980D84174a1ACb046,1.667273e+03,SHIB,ERC20,non traité,0.06112,NaN


In [12]:
source_node = hsds.find_source_node(chosen_group)

chosen_group, source_node = hsds.hashes_to_letters(filtered_chosen_group, source_node)

G, fig = hsds.draw_base_graph_four(filtered_chosen_group, source_node, create_using=nx.MultiDiGraph)

fig

In [27]:
print("Original Graph")
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
print(f"Number of nodes: {num_nodes}")
print(f"Number of edges (arrows): {num_edges}")

Original Graph
Number of nodes: 18
Number of edges (arrows): 17


In [13]:
graph = hsds.add_dummy_node_if_needed(G, source_node)

In [28]:
chosen_group = hsds.graph_to_dataframe(G)

In [29]:
chosen_group

,from,to,value,symbol,key
0,h,m,2.619030e+04,ECT,0
1,q,h,6.111984e-02,WETH,0
2,d,b,2.704797e+03,SHIB,0
3,d,c,1.869746e+03,SHIB,0
4,d,e,3.376653e+03,SHIB,0
5,d,f,2.339060e+03,SHIB,0
6,d,i,5.168041e+02,SHIB,0
7,d,k,6.842444e+02,SHIB,0
8,d,n,1.667273e+03,SHIB,0
9,d,o,1.271884e+04,SHIB,0


In [17]:
print("Graph with a Dummy node")
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
print(f"Number of nodes: {num_nodes}")
print(f"Number of edges (arrows): {num_edges}")

Graph with a Dummy node
Number of nodes: 19
Number of edges (arrows): 17


In [18]:
import gravis as gv

gv_defaults = dict( #From Jacky's utils:
    show_edge_label=True,
    edge_label_data_source="attr",
    edge_curvature=0.3,
    
    use_links_force=True,
    links_force_distance=30,
    links_force_strength=0.05,
    
    use_collision_force=True,
    collision_force_radius=50,
    collision_force_strength=.7,
    
    use_y_positioning_force=True,
    y_positioning_force_strength=0.1
)

fig = gv.d3(
        G, 
        **gv_defaults,
        #edge_label=edge_labels,  # Can gv.d3 use edge labels? NO! :/
    )

fig

In [34]:
graph = G
nodes_to_check = list(graph.nodes())
for node in nodes_to_check:
    if hsds.check_flow_conservation_node(graph, node):
        print(node)
        #print(hsds.check_flow_conservation_node(graph, node))
        hsds.split_flow_conservation_node(graph, node)   

print("Graph with flow_conservation_nodes splitted:")
# print(graph.edges(data=True))
num_nodes = graph.number_of_nodes()
num_edges = graph.number_of_edges()
print(f"Number of nodes: {num_nodes}")
print(f"Number of edges (arrows): {num_edges}")

Graph with flow_conservation_nodes splitted:
Number of nodes: 19
Number of edges (arrows): 17


In [ ]:
# All good up to now
# There is no flow conservation nodes

In [1]:
import importlib
importlib.reload(hsds)

graph = hsds.monochromatic_path_value(graph, source_node)
#  print("Processed Graph:")
# # print(graph.edges(data=True))
#  num_nodes = graph.number_of_nodes()
# #  num_edges = graph.number_of_edges()
#  print(f"Number of nodes: {num_nodes}")
#  print(f"Number of edges (arrows): {num_edges}")

NameError: name 'hsds' is not defined

In [40]:
chosen_group = hsds.graph_to_dataframe(graph)
chosen_group

,from,to,value,symbol,key
0,h,m,2.619030e+04,ECT,0
1,q,h,6.111984e-02,WETH,0
2,d,b,2.704797e+03,SHIB,0
3,d,c,1.869746e+03,SHIB,0
4,d,e,3.376653e+03,SHIB,0
5,d,f,2.339060e+03,SHIB,0
6,d,i,5.168041e+02,SHIB,0
7,d,k,6.842444e+02,SHIB,0
8,d,n,1.667273e+03,SHIB,0
9,d,o,1.271884e+04,SHIB,0


In [48]:
G, fig = hsds.draw_base_graph_four(chosen_group, source_node, create_using=nx.MultiDiGraph)

# fig

# fig = gv.d3(
#         graph, 
#         **gv_defaults,
#         #edge_label=edge_labels,  # Can gv.d3 use edge labels? NO! :/
#     )

fig

In [49]:
import importlib
importlib.reload(hsds)

graph = hsds.monochromatic_path_value(G, source_node)
#  print("Processed Graph:")
# # print(graph.edges(data=True))
#  num_nodes = graph.number_of_nodes()
# #  num_edges = graph.number_of_edges()
#  print(f"Number of nodes: {num_nodes}")
#  print(f"Number of edges (arrows): {num_edges}")

This is the node I am at: a
This is the node I am at: j
This is the first color: HKBY
This is the value: 4482133296.629102
This is the value: 4482133296.629102
I just marked this node: j
This is the node I am at: p
This is the value: 85160532635.95294
I just marked this node: p
This is the node I am at: g
The following node is marked: l
The following node is marked: j
There is a monochromatic path in this graph
The color (token) of this monochromatic path is HKBY
The sink in this monochromatic path is the node j
The value of the monochromatic path is 4482133296.629102
This is the node I am at: a
This is the node I am at: j
This is the node I am at: p
This is the first color: HKBY
This is the value: 85160532635.95294
This is the value: 85160532635.95294
I just marked this node: p
This is the node I am at: g
The following node is marked: l
The following node is marked: p
There is no monochromatic path in this graph.


In [50]:
type(graph)

NoneType

In [43]:
import importlib
importlib.reload(hsds)

chosen_group = hsds.graph_to_dataframe(graph)

G, fig = hsds.draw_base_graph_four(chosen_group, source_node, create_using=nx.MultiDiGraph)

AttributeError: 'NoneType' object has no attribute 'edges'

In [60]:
fig = gv.d3(
        graph, 
        **gv_defaults,
        #edge_label=edge_labels,  # Can gv.d3 use edge labels? NO! :/
    )

fig

In [69]:
import importlib
importlib.reload(hsds)

graph = hsds.monochromatic_path_value(graph, source_node)
#  print("Processed Graph:")
# # print(graph.edges(data=True))
#  num_nodes = graph.number_of_nodes()
# #  num_edges = graph.number_of_edges()
#  print(f"Number of nodes: {num_nodes}")
#  print(f"Number of edges (arrows): {num_edges}")

This is the node I am at: a
This is the first color: DIVIDEND_TRACKER
This is the value: 0.0
This is the node I am at: j
This is the node I am at: p
The following node is marked: l
There is no monochromatic path in this graph.


In [42]:
    # Step 5: Find an arc cut
status, marked_nodes, unmarked_nodes = hsds.find_arc_cut(graph, source_node)
    # print(f"Status: {status}")
    # print(f"Marked Nodes (S): {marked_nodes}")
    # print(f"Complement: {unmarked_nodes}")
    
    # Step 6: Sum WETH arcs
if marked_nodes is not None and unmarked_nodes is not None:
    # Apply the sum_weth_arcs function to sum the values of WETH arcs
        total_weth_value = hsds.sum_weth_arcs(graph, marked_nodes, unmarked_nodes)
    # print(f"Total WETH value: {total_weth_value}")
     
else:
        total_weth_value = -1

In [44]:
total_weth_value

-1

In [44]:
df[df['Connected_Components']>1]

,hash,source,from,to,value,symbol,sub_type,type,ETH_adj,transaction_fee,Transaction_Value,Connected_Components


In [46]:
len(df)

73341

In [47]:
df_swap = df[df['type'] == 'swap']
len(df_swap)

72770

In [48]:
len(df_swap) / len(df)

0.9922144503074679

In [55]:
def apply_tolerance(df, percentage_tolerance=1.0):
    # Filter rows where Transaction_Value > 0
    df_filtered = df[df['Transaction_Value'] > 0].copy()

    # Calculate tolerance in absolute terms for each row
    tolerance = df_filtered['ETH_adj'] * (percentage_tolerance / 100)

    # Create a column for tolerance and check the absolute difference
    df_filtered['Tolerance'] = tolerance
    df_filtered['Difference'] = abs(df_filtered['ETH_adj'] - df_filtered['Transaction_Value'])
    
    # Define matching and non-matching rows based on tolerance
    df_different = df_filtered[df_filtered['Difference'] > df_filtered['Tolerance']]
    df_same = df_filtered[df_filtered['Difference'] <= df_filtered['Tolerance']]

    return df_different, df_same

# Calculate and print the DataFrames with a 1% tolerance
df_different, df_same = apply_tolerance(df_swap, percentage_tolerance=1)

print("DataFrame Swap with Differences Exceeding Tolerance:")
print(len(df_different))

print("\nDataFrame Swap with Matches Within Tolerance:")
print(len(df_same))

DataFrame Swap with Differences Exceeding Tolerance:
25782

DataFrame Swap with Matches Within Tolerance:
41051


In [56]:
len(df_different) + len(df_same)

66833

In [57]:
len(df_same) / (len(df_different) + len(df_same))

0.6142324899376057

In [ ]:
# import helen4 as h4

# # Convert it to a MultiDiGraph
# multi_digraph = nx.MultiDiGraph(graph)

# total_weth_value = h4.main(multi_digraph, source_node)
# print(total_weth_value)

In [ ]:
# print(len(chosen_group["hash"])) #47 rows -> 46 EDGES ??????
# print(len(chosen_group["to"].unique()))
# print(len(chosen_group["from"].unique()))
# G , fig = make_base_graph(chosen_group, source_node)
# print(G.nodes)
# print(len(G.nodes))
# print(len(G.edges))

In [ ]:
# # Create an empty MultiDiGraph
# multi_digraph = nx.MultiDiGraph()

# # Add all the nodes from the original graph
# multi_digraph.add_nodes_from(G.nodes(data=True))

# # Manually add edges to the MultiDiGraph
# for u, v, data in G.edges(data=True):
#     # We can choose which direction to add the edge (u -> v or v -> u)
#     multi_digraph.add_edge(u, v, **data)

# # Verify the result
# print("Number of nodes:", multi_digraph.number_of_nodes())
# print("Number of edges (arrows):", multi_digraph.number_of_edges())

In [ ]:
chosen_hash

In [ ]:
# nulls = df_pandas['to'].isnull()
# df_pandas[nulls]
# # More problems !
# # Maybe delete this hash ?

In [ ]:
df_cleaned.head()

In [ ]:
df_grouped = df_cleaned.groupby("hash")

In [ ]:
hashes = df_cleaned["hash"].unique()
len(hashes)

In [ ]:
# Make sure there is no duplicates - REPETITION

# List all column names except 'source'
columns_to_compare = [col for col in df_cleaned.columns if col != 'source']

# Drop duplicates based on the columns to compare, keeping the first occurrence
df = df_cleaned.drop_duplicates(subset=columns_to_compare, keep='first')

In [ ]:
hashes = df["hash"].unique()
len(hashes)

In [ ]:
# So, df = df_cleaned, we are good!

In [ ]:
# In theory, I am recovering 73341 transactions.

In [ ]:
# import helen4 as h4

# groups = df_cleaned.groupby('hash')

# series = groups.apply(hsds.count_connected_components)

In [ ]:
# # Examples of hashes that have more than one connected component:

# import helen4 as h4
# from tqdm import tqdm

# countplus = 0
# count1 = 0

# # Wrapping the hashes list with tqdm for a progress bar
# for current_hash in tqdm(hashes, desc="Processing Hashes"):
#     df_current_hash = df_cleaned[df_cleaned["hash"] == current_hash]
#     G = create_graph_from_dataframe(df_current_hash)
    
#     if count_connected_components(G, mode='weak') > 1:
#         countplus += 1
#         print(f"The graph for the hash {current_hash} has more than 1 connected component")
#     else:
#         count1 += 1

# print(f"There were {countplus} hashes with more than 1 connected component and {count1} hashes with 1 connected component")


In [ ]:
# # Look for graphs with lots (>50?) of nodes:

# # chosen_hash = hashes[0]

# def node_count(chosen_hash):

#     chosen_group = df_grouped.get_group(chosen_hash)

#     source_node = find_source_node(chosen_group, chosen_hash)

#     graph, fig = make_base_graph(chosen_group, source_node)

#     return len(graph.nodes)

# for chosen_hash in tqdm(hashes, desc="Processing Hashes"):
#     if node_count(chosen_hash) > 50:
#         print(f"The hash {chosen_hash} gives us a graph with {node_count(chosen_hash)} nodes")

In [ ]:
# The hash 0x55dd24183ac9714b6aa75612e2a5f6b498831d901d4ac525cf920c0ccd9ed442 gives us a graph with 60 nodes
# The hash 0x9161743bec7180a169054036fd67fcceff4bb871066a8d20508ce0498e489cef gives us a graph with 63 nodes
# The hash 0x4372f75f874f0f4244ec191dea17bb2b75d45c607d894fd426e8d771efd1619f gives us a graph with 51 nodes
# Processing Hashes: 100%|██████████| 73341/73341 [25:32<00:00, 47.86it/s]


In [ ]:
len(hashes)


for hash in hashes:
    chosen_group = df_grouped.get_group(hash)
    if len(chosen_group) < 10:
        print(f"This hash's dataframe has less than 10 rows:", hash)

In [ ]:
import helen0822 as hsds

for chosen_hash in hashes:
    #print(f"the chosen hash is", chosen_hash)

    chosen_group = df_grouped.get_group(chosen_hash)
    # #print(f"the chosen group is", chosen_group)

    chosen_group_df = pd.DataFrame(chosen_group)
    # print(f"the lenght of the chosen_group_df is", len(chosen_group_df))

    # source_node = hsds.find_source_node(chosen_group_df)
    # #print(f"the source node is", source_node)

    # graph, fig = hsds.draw_base_graph(chosen_group_df, source_node)

    # G = hsds.create_graph(chosen_group_df)

    total_weth_value = hsds.main(chosen_group_df)
    print(total_weth_value)

In [ ]:
fig

In [ ]:
print(len(hashes))